# Contextual Retrieval Sandbox

Test the custom Contextual Retrieval framework implementation.

**Features:**
- Contextual embeddings (chunk-specific context via LLM)
- Hybrid search (vector + BM25)
- Reciprocal Rank Fusion
- Azure Cohere reranking

**Based on:** Anthropic's Contextual Retrieval research

In [1]:
# Setup: Load environment and imports
import os
import sys
from pathlib import Path
from pprint import pprint

# Add backend to path
backend_path = Path.cwd().parent.parent / "backend" / "src"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / ".env")

# Verify API keys
print("Environment check:")
print(f"  OPENAI_API_KEY: {'set' if os.getenv('OPENAI_API_KEY') else 'NOT SET'}")
print(f"  CO_API_KEY: {'set' if os.getenv('CO_API_KEY') else 'NOT SET'}")
print(f"  CO_RERANK_ENDPOINT: {os.getenv('CO_RERANK_ENDPOINT', 'not set')}")

Environment check:
  OPENAI_API_KEY: set
  CO_API_KEY: set
  CO_RERANK_ENDPOINT: https://mbaistudio3062596349.services.ai.azure.com/providers/cohere/v2/rerank


In [31]:
# Import contextual retrieval components
from uu_backend.services.contextual_retrieval import (
    ContextualRetrievalService,
    Chunk,
    ContextualizedChunk,
    SearchResult,
)
from uu_backend.services.contextual_retrieval.chunker import DocumentChunker
from uu_backend.services.contextual_retrieval.contextualizer import ChunkContextualizer
from uu_backend.services.contextual_retrieval.embedder import OpenAIEmbedder
from uu_backend.services.contextual_retrieval.vector_store import ChromaVectorStore
from uu_backend.services.contextual_retrieval.bm25_index import BM25Index
from uu_backend.services.contextual_retrieval.reranker import AzureCohereReranker, NoReranker
from uu_backend.services.contextual_retrieval.retriever import HybridRetriever

print("All imports successful!")

All imports successful!


## Load PDF Document

In [3]:
# Load PDF document
import pdfplumber

PDF_PATH = Path(r"C:\Projects\10k\intc.pdf")
DOC_ID = "intel_10k"

print(f"Loading PDF: {PDF_PATH}")
print(f"File exists: {PDF_PATH.exists()}")

# Extract text from PDF
def extract_pdf_text(pdf_path: Path) -> str:
    """Extract text from all pages of a PDF."""
    text_parts = []
    
    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total pages: {len(pdf.pages)}")
        
        for i, page in enumerate(pdf.pages):
            page_text = page.extract_text() or ""
            text_parts.append(page_text)
            
            if (i + 1) % 50 == 0:
                print(f"  Processed {i + 1} pages...")
    
    return "\n\n".join(text_parts)

DOCUMENT_CONTENT = extract_pdf_text(PDF_PATH)

print(f"\nExtracted {len(DOCUMENT_CONTENT):,} characters")
print(f"Preview (first 500 chars):\n{DOCUMENT_CONTENT[:500]}...")

Loading PDF: C:\Projects\10k\intc.pdf
File exists: True
Total pages: 270
  Processed 50 pages...
  Processed 100 pages...
  Processed 150 pages...
  Processed 200 pages...
  Processed 250 pages...

Extracted 928,373 characters
Preview (first 500 chars):
UNITED STATES SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 28, 2024.
or
☐
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from to .
Commission File Number: 000-06217
INTEL CORPORATION
(Exact name of registrant as specified in its charter)
Delaware 94-1672743
(State or other ju...


In [4]:
# Test chunking on the PDF document
chunker = DocumentChunker(chunk_size=1500, chunk_overlap=200)
chunks = chunker.chunk(DOC_ID, DOCUMENT_CONTENT)

print(f"Generated {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(c.text) for c in chunks) / len(chunks):.0f} chars")

# Show first few chunks
print("\n--- Sample Chunks ---")
for chunk in chunks[:3]:
    print(f"\nChunk {chunk.index} ({len(chunk.text)} chars):")
    print(f"  {chunk.text[:200]}...")

Generated 806 chunks
Average chunk size: 1252 chars

--- Sample Chunks ---

Chunk 0 (1498 chars):
  UNITED STATES SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ...

Chunk 1 (1400 chars):
  Indicate by check mark whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months (or for such
s...

Chunk 2 (1399 chars):
  provided pursuant to Section 13(a) of the Exchange Act. ☐
Indicate by check mark whether the registrant has filed a report on and attestation to its management's assessment of the effectiveness of its...


## Test Full Service Pipeline

In [5]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
CHROMA_DIR = DATA_DIR / "chroma_test"
BM25_PATH = DATA_DIR / "bm25_test.pkl"

# Create test directories
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print(f"ChromaDB directory: {CHROMA_DIR}")
print(f"BM25 index path: {BM25_PATH}")

ChromaDB directory: c:\Projects\test\data-labeller\data\chroma_test
BM25 index path: c:\Projects\test\data-labeller\data\bm25_test.pkl


In [32]:
# Initialize the service with settings optimized for 10-K documents
service = ContextualRetrievalService(
    chunk_size=1500,
    chunk_overlap=200,
    chroma_persist_directory=str(CHROMA_DIR),
    chroma_collection_name="intel_10k_collection",
    bm25_index_path=str(BM25_PATH),
    context_model="gpt-5-mini",
    embedding_model="text-embedding-3-small",
    use_reranking=True,
)

print("ContextualRetrievalService initialized!")
print(f"Stats: {service.get_stats()}")

ContextualRetrievalService initialized!
Stats: {'vector_store_count': 850, 'bm25_index_count': 967, 'reranker_type': 'AzureCohereReranker'}


In [11]:
# Index the Intel 10-K document (async with concurrency)
# Uses parallel API calls with retry/backoff for rate limits

import time

last_update = [0]
def progress_callback(stage, current, total):
    now = time.time()
    if now - last_update[0] > 0.3 or current == total:
        last_update[0] = now
        pct = current / total * 100 if total > 0 else 0
        filled = int(40 * current / total) if total > 0 else 0
        bar = "=" * filled + "-" * (40 - filled)
        print(f"\r{stage}: [{bar}] {current}/{total} ({pct:.0f}%)", end="", flush=True)
        if current == total:
            print()

print(f"Indexing {DOC_ID}...")
print(f"Document size: {len(DOCUMENT_CONTENT):,} characters")
print(f"Concurrency: {service.contextualizer.max_concurrency} parallel API calls")
print("Using async + retry with backoff\n")

start_time = time.time()

chunks_indexed = service.index_document(
    document_id=DOC_ID,
    content=DOCUMENT_CONTENT,
    metadata={"filename": PDF_PATH.name, "file_type": "pdf"},
    progress_callback=progress_callback,
)

elapsed = time.time() - start_time

# Show results and cache stats
cache_stats = service.contextualizer.get_cache_stats()
print(f"\n--- Results ---")
print(f"Indexed {chunks_indexed} chunks in {elapsed:.1f} seconds")
print(f"Throughput: {chunks_indexed / elapsed:.1f} chunks/sec")
print(f"\nPrompt Cache Stats (OpenAI):")
print(f"  Total prompt tokens: {cache_stats['total_prompt_tokens']:,}")
print(f"  Cached tokens: {cache_stats['total_cached_tokens']:,}")
print(f"  Cache hit rate: {cache_stats['cache_hit_rate']:.1f}%")
print(f"\nService stats: {service.get_stats()}")

Indexing intel_10k...
Document size: 928,373 characters
Concurrency: 5 parallel API calls
Using async + retry with backoff

contextualizing: [========--------------------------------] 163/806 (20%)

Retrying uu_backend.services.contextual_retrieval.contextualizer.ChunkContextualizer.acontextualize.<locals>._call_api in 2 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-WexwSOWxkRpjZAZxZtbZZhKI on tokens per min (TPM): Limit 500000, Used 496185, Requested 13003. Please try again in 1.102s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}.


contextualizing: [========--------------------------------] 166/806 (21%)

Retrying uu_backend.services.contextual_retrieval.contextualizer.ChunkContextualizer.acontextualize.<locals>._call_api in 2 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-WexwSOWxkRpjZAZxZtbZZhKI on tokens per min (TPM): Limit 500000, Used 496491, Requested 12997. Please try again in 1.138s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}.


contextualizing: [========--------------------------------] 173/806 (21%)

...

contextualizing: [=======================================-] 797/806 (99%)

Retrying uu_backend.services.contextual_retrieval.contextualizer.ChunkContextualizer.acontextualize.<locals>._call_api in 2 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-WexwSOWxkRpjZAZxZtbZZhKI on tokens per min (TPM): Limit 500000, Used 495447, Requested 12993. Please try again in 1.012s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}.


contextualizing: [=======================================-] 802/806 (100%)

Retrying uu_backend.services.contextual_retrieval.contextualizer.ChunkContextualizer.acontextualize.<locals>._call_api in 2 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-WexwSOWxkRpjZAZxZtbZZhKI on tokens per min (TPM): Limit 500000, Used 495636, Requested 12831. Please try again in 1.016s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}.


contextualizing: [========================================] 806/806 (100%)
storing: [----------------------------------------] 0/806 (0%)
--- Results ---
Indexed 806 chunks in 951.2 seconds
Throughput: 0.8 chunks/sec

Prompt Cache Stats (OpenAI):
  Total prompt tokens: 8,093,416
  Cached tokens: 7,326,464
  Cache hit rate: 90.5%

Service stats: {'vector_store_count': 850, 'bm25_index_count': 967, 'reranker_type': 'NoReranker'}


In [33]:
# Test search with 10-K relevant queries
TEST_QUERIES = [
    "Cash from Operating Activities $B",
]

print("Testing search queries on Intel 10-K...")
print("=" * 60)

for query in TEST_QUERIES:
    print(f"\nQuery: {query}")
    print("-" * 40)
    
    results = service.search(query, top_k=3)
    
    if results:
        for i, result in enumerate(results):
            print(f"\n  Result {i+1} (score: {result.score:.4f}):")
            print(f"    Context: {result.context[:150]}...")
            print(f"    Text preview: {result.original_text[:200]}...")
    else:
        print("  No results found")

print("\n" + "=" * 60)
print("Search testing complete!")

Testing search queries on Intel 10-K...

Query: Cash from Operating Activities $B
----------------------------------------

  Result 1 (score: 0.8498):
    Context: ...
    Text preview: Financial Capital
We take a disciplined approach to our financial capital allocation strategy, which continues to focus on building stakeholder value and is driven by our priority to invest in the
bus...

  Result 2 (score: 0.8135):
    Context: This section outlines Intel's financial capital allocation strategy—prioritizing R&D and capital spending under its Smart Capital approach while suspe...
    Text preview: Financial Capital
We take a disciplined approach to our financial capital allocation strategy, which continues to focus on building stakeholder value and is driven by our priority to invest in the
bus...

  Result 3 (score: 0.7292):
    Context: Excerpt from the Financial Statements’ Consolidated Statements of Cash Flows showing operating and investing cash flow items for fiscal years ended 2

## Multi-Query Search for Extraction

Use multiple queries to find all relevant chunks for a schema.

In [20]:
api_key

'0dde9bfd4a914b43818e64b1e18e7d2e'

In [22]:
# Try without model parameter - Azure may auto-select
payload_no_model = {
    "query": "What is Intel's revenue?",
    "documents": ["Intel revenue was $54B.", "Chips are made."],
    "top_n": 2,
}

print("Testing WITHOUT model parameter...")
r = requests.post(endpoint, headers=headers, json=payload_no_model, timeout=30)
print(f"Status: {r.status_code}")
print(f"Response: {r.text[:500]}")

Testing WITHOUT model parameter...
Status: 500
Response: { "statusCode": 500, "message": "Internal server error", "activityId": "c7a69e1f-deaf-4db4-a886-9ac591ca50b6" }


In [27]:
# Try using Cohere SDK with Azure endpoint
# pip install cohere if needed

try:
    import cohere
    
    co = cohere.ClientV2(
        api_key=os.getenv("CO_API_KEY"),
        base_url=os.getenv("CO_RERANK_ENDPOINT").replace("/v2/rerank", ""),
    )
    
    result = co.rerank(
        query="What is Intel's revenue?",
        documents=["Intel revenue was $54B.", "Chips are made."],
        top_n=2,
    )
    print("Success!")
    print(result)
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Error: TypeError: V2Client.rerank() missing 1 required keyword-only argument: 'model'


In [29]:
# Use the correct model from Azure: Cohere-rerank-v4.0-fast
import requests

endpoint = "https://mbaistudio3062596349.services.ai.azure.com/providers/cohere/v2/rerank"
api_key = os.getenv("CO_API_KEY")

payload = {
    "model": "Cohere-rerank-v4.0-fast",
    "query": "What is Intel's revenue?",
    "documents": ["Intel revenue was $54B.", "Chips are made."],
    "top_n": 2,
}

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

print("Testing with Cohere-rerank-v4.0-fast...")
r = requests.post(endpoint, headers=headers, json=payload, timeout=30)
print(f"Status: {r.status_code}")
print(f"Response: {r.text}")

Testing with Cohere-rerank-v4.0-fast...
Status: 200
Response: {"id":"c43e9944-1c93-4483-bc87-54d58de161d3","results":[{"index":0,"relevance_score":0.88616455},{"index":1,"relevance_score":0.18808526}],"meta":{"api_version":{"version":"2"},"billed_units":{"search_units":1}}}


In [21]:
endpoint

'https://mbaistudio3062596349.services.ai.azure.com/providers/cohere/v2/rerank'

In [19]:
# Test Azure Cohere Reranker directly
import requests

endpoint = os.getenv("CO_RERANK_ENDPOINT")
api_key = os.getenv("CO_API_KEY")

print(f"Endpoint: {endpoint}")
print(f"API Key set: {bool(api_key)}")

# Test payload
payload = {
    "model": "Cohere-rerank-v3.5-turbo-128k",
    "query": "What is Intel's revenue?",
    "documents": [
        "Intel reported revenue of $54 billion for fiscal year 2024.",
        "The company manufactures semiconductor chips.",
        "Intel was founded in 1968.",
    ],
    "top_n": 3,
}

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

print("\nTesting rerank API...")
response = requests.post(endpoint, headers=headers, json=payload, timeout=30)

print(f"Status: {response.status_code}")
print(f"Response: {response.text[:1000]}")

Endpoint: https://mbaistudio3062596349.services.ai.azure.com/providers/cohere/v2/rerank
API Key set: True

Testing rerank API...
Status: 500
Response: { "statusCode": 500, "message": "Internal server error", "activityId": "58c197af-ad54-4b06-a78b-7f32805fcc73" }


In [37]:
# RAG Answer Generation
from openai import OpenAI

client = OpenAI()

def generate_answer(query: str, results: list, model: str = "gpt-5-mini") -> str:
    """Generate an answer using retrieved chunks as context."""
    
    # Format chunks as context
    context_parts = []
    for i, r in enumerate(results):
        context_parts.append(f"[Source {i+1}]\n{r.original_text}")
    
    context = "\n\n".join(context_parts)
    
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer the user's question based on the provided context. If the answer is not in the context, say so."
        },
        {
            "role": "user", 
            "content": f"Context:\n{context}\n\n---\n\nQuestion: {query}"
        }
    ]
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=500,
    )
    
    return response.choices[0].message.content

# Test it
query = "What does semiconductor manufacturing mean"
results = service.search(query, top_k=5)

print(f"Query: {query}\n")
print("=" * 60)
answer = generate_answer(query, results)
print(f"\nAnswer:\n{answer}")

Query: What does semiconductor manufacturing mean


Answer:
According to the provided context, “Semiconductor Manufacturing” means semiconductor wafer production, semiconductor fabrication, or semiconductor packaging.

- Semiconductor wafer production: processes such as wafer slicing, polishing, cleaning, epitaxial deposition, and metrology.  
- Semiconductor fabrication: forming devices on a wafer (e.g., transistors, poly capacitors, non‑metal resistors, diodes).  
- Semiconductor packaging: (assembly and packaging of finished semiconductor dies or wafers).


## Cleanup

In [38]:
# Delete indexed document (optional - run only if you want to clear the index)
# result = service.delete_document(DOC_ID)
# print(f"Deleted document: {result}")
# print(f"Stats after deletion: {service.get_stats()}")

print("To delete the indexed document, uncomment the lines above and run this cell.")
print(f"Current stats: {service.get_stats()}")

To delete the indexed document, uncomment the lines above and run this cell.
Current stats: {'vector_store_count': 850, 'bm25_index_count': 967, 'reranker_type': 'AzureCohereReranker'}
